
# Scenario overview

Reads a scenario's **setup** straight from the intent database -- no run
required. Complements [accounts.ipynb](./accounts.ipynb) (account detail)
and [net_worth.ipynb](./net_worth.ipynb) (simulation results): this notebook
is about what was *authored* (manifest, persons, accounts, scheduled
transactions, chains, calendar events), not what the engine produced.

Pick a scenario below, then re-run the cells below.

> **Table clipped, or the notebook slow?** Run `analysis jupyter` and open this notebook in JupyterLab: wide tables scroll sideways there, and it can use less memory than VS Code. The terminal shows the same information without a notebook: `analysis sim account list`, `analysis sim scheduled list` and `analysis sim transfer list`.

In [ ]:
%load_ext autoreload
%autoreload 2

from analysis.sim.notebook_ui.pickers import scenario_picker
from analysis.sim.notebook_ui.scenario_view import persons_view
from analysis.sim.notebook_ui.scenario_plots import show_chains, show_chain_flow, show_calendar_view, show_scheduled_view
from analysis.sim.notebook_ui.accounts_plots import show_account_setup


# Pick a different scenario here, then re-run this cell and the ones below.
# CONNECTIONS.scenario is a live ScenarioConnection -- this notebook never
# needs a run, so `require_run=False`.
PICKER, CONNECTIONS, SCENARIO_PICKER = scenario_picker(require_run=False,
                                                       show_metadata=True,
                                                       show_description=True,
                                                       default="base")
display(PICKER)

## Persons

Everyone that this scenario's accounts, scheduled transactions, and lifespan
rules can reference.


In [ ]:
CONNECTIONS.show_scenario(persons_view)

## Accounts

Same view as [accounts.ipynb](./accounts.ipynb)'s rich table: kind, owner,
tax treatment, opening balance, and applicable growth/yield rate.


In [ ]:
CONNECTIONS.show_scenario(show_account_setup)

## Chains

A **chain** is a named, ordered list of places money comes from or goes to. Where a rule
needs a source or destination, a chain lets it say "try this, then that" — draw from
savings first, then an IRA — instead of every rule repeating the same ordering. Each step
can take a share (`percent`) or a cap (`maximum`) of what is still needed, and whatever it
doesn't supply falls through to the next step. A lot (an account's position) can also name a chain to top
itself up (`refill_chain`) or to send its excess to (`overflow_chain`).

The table lists each chain's steps in order. `draw_from` is an account or another chain;
`lot` says which lot of that account, and is `-` when the account has just one.

Details: [ADR-003 §4.7](../docs/architecture/ADR-003-registries-and-csv-schemas.md).


In [ ]:
CONNECTIONS.show_scenario(show_chains)

One chain at a time as a flowchart, picked from the list. A numbered line is a step of the chain it hangs from, with
the share it takes; a dotted arrow is a lot's refill (before a debit would take it below its
minimum), a thick arrow its overflow (above its maximum, at month end), and a plain arrow a
rule drawing from or paying into a chain. `sim chain show CHAIN --mermaid` prints the same chart, and
[the chains guide](../docs/guides/chains.md) explains each piece.


In [ ]:
CONNECTIONS.show_scenario(show_chain_flow)

## Scheduled transactions

Every recurring rule, in the order the engine evaluates them (priority,
then name -- ADR-001's priority bands).


In [ ]:
CONNECTIONS.show_scenario(show_scheduled_view)

## Calendar events

One-time interventions (`set_amount`, retirements, benefit cuts, etc.), in
date order.


In [ ]:
CONNECTIONS.show_scenario(show_calendar_view)